# Faster full-spectrum correction

Correct the entire bundled HARPS spectrum twice, then compare runtime and fit residuals. No cropping, stellar template, or saved outputs. Run the cells in order; the widget toolbar allows zooming.

The first run may download and cache atmospheric data. Rerun both correction cells before judging the speed ratio. Timings exclude loading the FITS file and plotting.

## 1. Default correction

In [ ]:
%matplotlib widget
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np

from pymolfit import AdvancedOptions, correct, load_spectrum

data = Path("docs/user_guide/data")
if not data.is_dir():
    data = Path("../user_guide/data")
fits = data / "ADP.2017-04-07T01_04_41.632.fits"
spectrum = load_spectrum(fits, wavelength_medium="air", save_header=False)

print(f"Default correction: {spectrum.flux.size:,} input pixels", flush=True)
start = perf_counter()
standard = correct(spectrum, report=False)
standard_seconds = perf_counter() - start
print(f"Default: {standard_seconds:.1f} s; converged: {standard.success}")
if not standard.success:
    print(standard.message)

## 2. Choose fit settings explicitly

Skip automatic comparisons of continuum orders, wavelength models, and instrumental profiles, and the optional robust-loss-scale refit. The molecular scales, continuum coefficients, wavelength coefficients, and Gaussian width are still fitted. Automatic fitting-region selection and the atmospheric calculation are unchanged.

| Setting | Choice |
| --- | --- |
| `continuum_order=2` | Quadratic continuum in each segment. |
| `wavelength_order=1` | Linear wavelength correction. |
| `fit_lsf_lorentz_fwhm=False` | Gaussian instrumental profile, without fitted Lorentzian wings. |
| `lsf_variable_width=True` | Width proportional to wavelength, rather than selecting a width law. Appropriate to test for approximately constant resolving power on this uniformly spaced wavelength grid. |
| `f_scale=1.0` | Keep the soft-L1 transition at one flux uncertainty instead of estimating it and refitting. This downweights large residuals more strongly than a larger scale; it does not rescale the uncertainties. |

These assumptions can save time but are not universally suitable. Compare the residuals before adopting them for other observations. Solver tolerances and the soft-L1 loss function retain their defaults.

In [ ]:
fast_options = AdvancedOptions(
    continuum_order=2,
    wavelength_order=1,
    fit_lsf_lorentz_fwhm=False,
    lsf_variable_width=True,
    f_scale=1.0,
)

print("Correction with faster settings", flush=True)
start = perf_counter()
faster = correct(spectrum, advanced=fast_options, report=False)
faster_seconds = perf_counter() - start
print(f"Faster settings: {faster_seconds:.1f} s; converged: {faster.success}")
if standard.success and faster.success:
    print(f"Runtime ratio (default / faster): {standard_seconds / faster_seconds:.2f}x")
else:
    print("At least one fit did not converge; inspect its message before comparing quality.")
    print(standard.message, faster.message, sep="\n")

## 3. Compare spectra and residuals

The upper panel overlays the uncorrected spectrum and both corrections over the full coverage. The lower panels show $(F^{\mathrm{obs}}-F^{\mathrm{mod}})/\sigma_F$ on the same valid fitting pixels, using the default run's flux uncertainties for both. Smaller absolute residuals indicate closer agreement with the observation, not an independent measure of telluric-correction accuracy.

Gaps in the residual panels mark pixels not used by both fits; they do not mean those wavelengths were left uncorrected. The two residual panels share their vertical scale.

In [ ]:
observed = standard.spectrum.to_unit("nm")
wave = observed.wavelength
np.testing.assert_allclose(wave, faster.spectrum.to_unit("nm").wavelength, rtol=0, atol=0)
sigma = observed.uncertainty
common = (
    observed.valid
    & faster.spectrum.valid
    & standard.fit_mask
    & faster.fit_mask
    & np.isfinite(standard.model_flux)
    & np.isfinite(faster.model_flux)
)
if not common.any():
    raise ValueError("The two corrections have no valid fitting pixels in common.")

fig, axes = plt.subplots(
    3,
    1,
    figsize=(12, 7),
    sharex=True,
    gridspec_kw={"height_ratios": [2, 1, 1]},
    layout="constrained",
)
axes[2].sharey(axes[1])
axes[0].plot(
    wave,
    np.where(observed.valid, observed.flux, np.nan),
    color="0.6",
    linewidth=0.8,
    label="Uncorrected",
)

for ax, result, label, color, style in (
    (axes[1], standard, "Default", "tab:orange", "-"),
    (axes[2], faster, "Faster settings", "tab:blue", "--"),
):
    axes[0].plot(
        wave,
        np.where(result.corrected.valid, result.corrected.flux, np.nan),
        color=color,
        linewidth=0.8,
        linestyle=style,
        label=label,
    )
    residual = np.full(wave.size, np.nan)
    residual[common] = (observed.flux[common] - result.model_flux[common]) / sigma[common]
    ax.plot(wave, residual, ".", markersize=1.5, color=color, label=label)
    ax.axhline(0, color="0.5", linewidth=0.7)
    ax.set_ylabel("Residual / uncertainty")
    ax.legend(loc="upper right")
    rms = np.sqrt(np.mean(residual[common] ** 2))
    print(f"{label}: residual RMS = {rms:.3f} on {common.sum():,} common fitting pixels")

axes[0].set_ylabel("Flux")
axes[0].legend(loc="upper right")
axes[-1].set_xlabel("Observatory-frame vacuum wavelength [nm]")
axes[-1].ticklabel_format(axis="x", useOffset=False)
plt.show()